# k-means clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> kmeans_clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from gapstat import gapstat_score
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn import clone
from sklearn.cluster import KMeans

from utils.bootstrap_utils import bootstrap, bootstrap_grid_search
from utils.clustering_utils import (
    predict_clustering_labels,
    predict_clustering_proba,
)
from utils.metrics_utils import (
    adjusted_rand_score,
    association,
    calinski_harabasz_score,
    davies_bouldin_score,
    dunn_index,
    normalized_mutual_info_score,
    silhouette_score,
    variation_of_information,
)
from utils.misc_utils import contingency_matrix
from utils.saving_utils import DatasetBundle, save_object

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

# Set outputs folder
outputs_folder = "outputs/kmeans_clustering"

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Create RNG for reproducibility
rng0 = np.random.default_rng(352532)
rng1, rng2 = rng0.spawn(2)
kmeans_discov = KMeans(random_state=np.random.RandomState(rng0.bit_generator))
kmeans_valid = KMeans(random_state=np.random.RandomState(rng1.bit_generator))

n_boot = 10
param_grid = {"n_clusters": range(2, 51)}
within_metrics = {
    "neg_sum_of_sq": lambda e, X, sample_weight: e.score(X, sample_weight=sample_weight),
    "dbi": lambda e, X, sample_weight: davies_bouldin_score(X, labels=e.predict(X), sample_weight=sample_weight),
    "chi": lambda e, X, sample_weight: calinski_harabasz_score(X, labels=e.predict(X), sample_weight=sample_weight),
    "si": lambda e, X, sample_weight: silhouette_score(X, labels=e.predict(X), sample_weight=sample_weight),
    "di": lambda e, X, sample_weight: dunn_index(X, labels=e.predict(X)),
}
between_metrics = {
    "ari": lambda x, y, sample_weight: adjusted_rand_score(x, y, sample_weight=sample_weight),
    "nmi": lambda x, y, sample_weight: normalized_mutual_info_score(x, y, sample_weight=sample_weight),
    "vi": lambda x, y, sample_weight: variation_of_information(x, y, sample_weight=sample_weight),
    "assoc": lambda x, y, sample_weight: association(
        contingency_matrix(x, y, sample_weight=sample_weight), correction=False
    ),
    "chi2": lambda x, y, sample_weight: (
        chi2_contingency(contingency_matrix(x, y, sample_weight=sample_weight), correction=False).statistic
    ),
}

# Run bootstrap grid-search loop
metrics_grid_search_discov = bootstrap_grid_search(
    kmeans_discov,
    data_discov_mfa.to_numpy(),
    param_grid,
    n_boot=n_boot,
    within_metrics=within_metrics,
    between_metrics=between_metrics,
    sample_weight=sample_weight_discov,
    rng=rng2,
)
smaller_is_better_metrics = ["dbi", "vi"]
bigger_is_better_metrics = ["neg_sum_of_sq", "chi", "si", "di", "ari", "nmi", "vi", "assoc", "chi2"]
metrics_grid_search_discov["bigger_is_better"] = metrics_grid_search_discov["metric"].map(
    lambda x: x in bigger_is_better_metrics
)

# Find and display best performing parameters
min_metrics = (
    metrics_grid_search_discov.loc[lambda x: ~x["bigger_is_better"]]
    .loc[lambda x: x.groupby("metric")["mean"].idxmin()]
    .sort_values("type")
)
max_metrics = (
    metrics_grid_search_discov.loc[lambda x: x["bigger_is_better"]]
    .loc[lambda x: x.groupby("metric")["mean"].idxmax()]
    .sort_values("type")
)
best_metrics = pd.concat([min_metrics, max_metrics])
print("minimum mean values:")
display(min_metrics)
print("maximum mean values:")
display(max_metrics)

In [ ]:
# Set best parameters and refit models
selection_metric = "ari"
best_params = (
    best_metrics[lambda x: x["metric"].eq(selection_metric)]
    .drop(columns=["metric", "mean", "std", "type", "bigger_is_better"])
    .to_dict(orient="records")[0]
)
print(f"Best parameters wrt. {selection_metric}: {best_params}")

kmeans_discov.set_params(**best_params)
kmeans_valid.set_params(**best_params)
model_discov = clone(kmeans_discov)
model_valid = clone(kmeans_valid)
model_discov.fit(data_discov_mfa, sample_weight=sample_weight_discov)
model_valid.fit(data_valid_mfa, sample_weight=sample_weight_valid)
None

In [ ]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_mfa)
labels_valid = predict_clustering_labels(model_valid, data_valid_mfa)

# Apply models to predict probabilities
proba_discov = predict_clustering_proba(model_discov, data_discov_mfa)
proba_valid = predict_clustering_proba(model_valid, data_valid_mfa)

### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Compare clusterings
metrics_valid = bootstrap(
    kmeans_valid,
    data_valid_mfa,
    n_boot=n_boot,
    within_metrics=within_metrics,
    between_metrics=between_metrics,
    sample_weight=sample_weight_valid,
    rng=np.random.default_rng(2352),
)
metrics_valid["bigger_is_better"] = metrics_valid["metric"].map(lambda x: x in bigger_is_better_metrics)

(
    metrics_grid_search_discov.loc[lambda x: np.logical_and.reduce([x[col] == val for col, val in best_params.items()])]
    .drop(columns=best_params)
    .set_index(["metric", "type", "bigger_is_better"])
    .join(metrics_valid.set_index(["metric", "type", "bigger_is_better"]), lsuffix="_discov", rsuffix="_valid")
    .reset_index()
)

metrics_discov_valid_comparison = (
    metrics_grid_search_discov.loc[lambda x: np.logical_and.reduce([x[col] == val for col, val in best_params.items()])]
    .drop(columns=best_params)
    .set_index(["metric", "type", "bigger_is_better"])
    .join(metrics_valid.set_index(["metric", "type", "bigger_is_better"]), lsuffix="_discov", rsuffix="_valid")
    .reset_index()
)
metrics_discov_valid_comparison

In [ ]:
# Save grid-search results
metrics_grid_search_discov.to_parquet(Path(outputs_folder, "metrics_grid_search_discov.parquet"), index=None)

# Save models
save_object(model_discov, Path(outputs_folder, "model_discov.pkl"))
save_object(model_valid, Path(outputs_folder, "model_valid.pkl"))

# Save predictions
proba_discov.assign(**{"label": labels_discov}).to_parquet(
    Path(outputs_folder, "cluster_pred_discov.parquet"), index=False
)
proba_valid.assign(**{"label": labels_valid}).to_parquet(
    Path(outputs_folder, "cluster_pred_valid.parquet"), index=False
)

# Save comparison
metrics_discov_valid_comparison.to_parquet(Path(outputs_folder, "metrics_discov_valid_comparison.parquet"), index=False)